# **Learn, Love, Live**

In [15]:
print("jai maa kali")

jai maa kali


In [16]:
# import pandas as pd
# import numpy as np
# from sklearn.preprocessing import OneHotEncoder, StandardScaler
# from sklearn.metrics.pairwise import cosine_similarity

# # Your dataframe 'df' is already loaded

# def simple_cosine_recommendation(df):
#     """Simplified version for cosine similarity recommendations"""
    
#     # Step 1: Aggregate exercise features
#     exercise_features = df.groupby('exercise_id').agg({
#         'muscle_group': 'first',
#         'body_part': 'first', 
#         'difficulty_level': 'first',
#         'equipment': 'first',
#         'experience_level': 'first',
#         'avg_completion_rate': 'mean',
#         'avg_rpe': 'mean',
#         'avg_sets_completed': 'mean'
#     }).reset_index()
    
#     # Step 2: One-hot encode categorical features
#     categorical_cols = ['muscle_group', 'body_part', 'difficulty_level', 
#                        'equipment', 'experience_level']
    
#     onehot_encoder = OneHotEncoder(sparse_output=False)
#     categorical_encoded = onehot_encoder.fit_transform(exercise_features[categorical_cols])
    
#     # Step 3: Scale numerical features
#     numerical_cols = ['avg_completion_rate', 'avg_rpe', 'avg_sets_completed']
#     scaler = StandardScaler()
#     numerical_scaled = scaler.fit_transform(exercise_features[numerical_cols])
    
#     # Step 4: Combine features
#     all_features = np.hstack([categorical_encoded, numerical_scaled])
    
#     # Step 5: Compute cosine similarity
#     similarity_matrix = cosine_similarity(all_features)
    
#     # Create readable DataFrame
#     similarity_df = pd.DataFrame(
#         similarity_matrix,
#         index=exercise_features['exercise_id'],
#         columns=exercise_features['exercise_id']
#     )
    
#     return similarity_df, exercise_features

# # Use it
# similarity_df, exercise_features = simple_cosine_recommendation(df)

# print("Exercise Similarity Matrix:")
# print(similarity_df.round(3))

# # Get similar exercises
# exercise_id = 45
# similar_exercises = similarity_df[exercise_id].sort_values(ascending=False)[1:4]  # Exclude self
# print(f"\nExercises similar to {exercise_id}:")
# for ex_id, score in similar_exercises.items():
#     print(f"  Exercise {ex_id}: similarity = {score:.3f}")

In [17]:
# understand or working with dataframe modules
import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings("ignore")


# model building modules
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.metrics.pairwise import cosine_similarity


# Display settings for pandas DataFrames
pd.set_option('display.max_columns', None)  # Show all columns
pd.set_option('display.max_rows', None)     # Show all rows
pd.set_option('display.max_colwidth', None) # Show full column content
pd.set_option('display.width', None)        # Auto-detect terminal width
pd.set_option('display.expand_frame_repr', False) # Don't wrap to multiple pages

## **Start**

In [18]:
gym = pd.read_csv("data/model_data/excersie_selection.csv")
gym

,user_id,exercise_id,muscle_group,body_part,difficulty_level,equipment,day_type,experience_level,session_date,times_shown,times_completed,times_skipped,avg_completion_rate,avg_rpe,avg_sets_completed,is_selected
0,101,45,quadriceps,legs,Intermediate,barbell,legs,intermediate,2025-01-01,5,4,1,0.95,7,3.8,1
1,101,52,hamstrings,legs,Intermediate,dumbbell,legs,intermediate,2025-01-01,3,1,2,0.55,8,2.1,0
2,101,60,glutes,legs,Beginner,body_weight,legs,intermediate,2025-01-01,2,2,0,1.00,6,3.0,1
3,102,12,abs,waist,Beginner,body_weight,abs,beginner,2025-01-02,4,4,0,1.00,5,2.5,1
4,102,18,abs,waist,Intermediate,body_weight,abs,beginner,2025-01-02,2,0,2,0.20,9,1.0,0
5,103,88,quadriceps,legs,Advanced,barbell,legs,advanced,2025-01-03,6,5,1,0.92,8,4.2,1


## **EDA**

In [19]:
gym.shape

(6, 16)

In [20]:
gym.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 6 entries, 0 to 5
Data columns (total 16 columns):
 #   Column               Non-Null Count  Dtype  
---  ------               --------------  -----  
 0   user_id              6 non-null      int64  
 1   exercise_id          6 non-null      int64  
 2   muscle_group         6 non-null      object 
 3   body_part            6 non-null      object 
 4   difficulty_level     6 non-null      object 
 5   equipment            6 non-null      object 
 6   day_type             6 non-null      object 
 7   experience_level     6 non-null      object 
 8   session_date         6 non-null      object 
 9   times_shown          6 non-null      int64  
 10  times_completed      6 non-null      int64  
 11  times_skipped        6 non-null      int64  
 12  avg_completion_rate  6 non-null      float64
 13  avg_rpe              6 non-null      int64  
 14  avg_sets_completed   6 non-null      float64
 15  is_selected          6 non-null      int64  

In [21]:
# list of which are not using in model building
unused_columns = ['user_id', 'session_date']
clean_Df = gym.drop(columns=unused_columns)
clean_Df.head()

,exercise_id,muscle_group,body_part,difficulty_level,equipment,day_type,experience_level,times_shown,times_completed,times_skipped,avg_completion_rate,avg_rpe,avg_sets_completed,is_selected
0,45,quadriceps,legs,Intermediate,barbell,legs,intermediate,5,4,1,0.95,7,3.8,1
1,52,hamstrings,legs,Intermediate,dumbbell,legs,intermediate,3,1,2,0.55,8,2.1,0
2,60,glutes,legs,Beginner,body_weight,legs,intermediate,2,2,0,1.00,6,3.0,1
3,12,abs,waist,Beginner,body_weight,abs,beginner,4,4,0,1.00,5,2.5,1
4,18,abs,waist,Intermediate,body_weight,abs,beginner,2,0,2,0.20,9,1.0,0


## **Model**

In [22]:
model_Df = clean_Df.groupby('exercise_id').agg({
    "muscle_group": "first",
    "body_part": "first",
    "difficulty_level": "first",
    "equipment": "first",
    "day_type": "first",
    "experience_level": "first",
    "times_shown": "mean",
    "times_completed": "mean",
    "times_skipped": "mean",
    "avg_completion_rate": "mean",
    "avg_rpe": "mean",
    "avg_sets_completed": "mean",
    "is_selected": "first",
}).reset_index()

model_Df.head()

,exercise_id,muscle_group,body_part,difficulty_level,equipment,day_type,experience_level,times_shown,times_completed,times_skipped,avg_completion_rate,avg_rpe,avg_sets_completed,is_selected
0,12,abs,waist,Beginner,body_weight,abs,beginner,4.0,4.0,0.0,1.00,5.0,2.5,1
1,18,abs,waist,Intermediate,body_weight,abs,beginner,2.0,0.0,2.0,0.20,9.0,1.0,0
2,45,quadriceps,legs,Intermediate,barbell,legs,intermediate,5.0,4.0,1.0,0.95,7.0,3.8,1
3,52,hamstrings,legs,Intermediate,dumbbell,legs,intermediate,3.0,1.0,2.0,0.55,8.0,2.1,0
4,60,glutes,legs,Beginner,body_weight,legs,intermediate,2.0,2.0,0.0,1.00,6.0,3.0,1


In [ ]:
# here we are using one hot coding for encode the categorical columns
object_cols = model_Df.select_dtypes(include=['object']).columns
onehot_encoder = OneHotEncoder(sparse_output=False, drop='first')
object_encoded = onehot_encoder.fit_transform(model_Df[object_cols])


In [24]:
# here we are using scaler encoding for scale the numerical columns
numerical_cols = model_Df.select_dtypes(include=['int64', 'float64']).columns
scaler_encoder = StandardScaler()
numerical_scaled = scaler_encoder.fit_transform(model_Df[numerical_cols])


In [25]:
# here we are combine the both object and numerical data in one for model buidling
all_features = np.hstack((object_encoded, numerical_scaled))
all_features.shape


(6, 19)

In [26]:
# build the model 
similarity_matrix = cosine_similarity(all_features)

In [27]:
similarity_Df = pd.DataFrame(similarity_matrix, index=model_Df['exercise_id'], columns=model_Df['exercise_id'])
similarity_Df.head()

exercise_id,12,18,45,52,60,88
exercise_id,,,,,,
12,1.000000,-0.173966,0.188232,-0.445527,0.432822,-0.111778
18,-0.173966,1.000000,-0.413185,0.629927,-0.231410,-0.568221
45,0.188232,-0.413185,1.000000,-0.025514,0.234079,0.689556
52,-0.445527,0.629927,-0.025514,1.000000,-0.072910,-0.205834
60,0.432822,-0.231410,0.234079,-0.072910,1.000000,0.029456


In [28]:
excerise_id = 45
similar_exercises = similarity_Df[excerise_id].sort_values(ascending=False)[1:4]
for ex_id, score in similar_exercises.items():
    print(f"  Exercise {ex_id}: similarity = {score:.3f}")
    value = model_Df[model_Df['exercise_id'] == ex_id][object_cols]
    print(f"{value}\n")

  Exercise 88: similarity = 0.690
  muscle_group body_part difficulty_level equipment day_type experience_level
5   quadriceps      legs         Advanced   barbell     legs         advanced

  Exercise 60: similarity = 0.234
  muscle_group body_part difficulty_level    equipment day_type experience_level
4       glutes      legs         Beginner  body_weight     legs     intermediate

  Exercise 12: similarity = 0.188
  muscle_group body_part difficulty_level    equipment day_type experience_level
0          abs     waist         Beginner  body_weight      abs         beginner

